In [8]:
#TODO make sure this renders in the github repo

# Rotary Positional Embeddings (RoPE)


![RoPE Diagram](../showcase_images/RoPE.png)

**From Llama 1 paper:** "Rotary Embeddings [GPTNeo]. We remove the absolute positional embeddings, and instead, add rotary positional embeddings (RoPE), introduced by [Su et al. (2021)](https://arxiv.org/abs/2104.09864), at each layer of the network."

- The "absolute positional embeddings" was the Positional Encoding in the Transformer architecture from Attention is All You Need paper.

**From Llama 3 paper:** All the Llama 3 models used $\text{Positional Embeddings} = RoPE (\theta =500{,}000)$ from Table 3 Overview of the key hyperparameters of Llama 3.


RoPE rotates embeddings of the **Query** and **Key** tensors by $\theta$ radians based on their position in the sequence.

- RoPE encodes the physical sequence position.
- The difference in angle tells you exactly how many steps apart two words are in the sentence structure.
  ![RoPE rotation](../showcase_images/RoPE_rotation_ex.png)
- RoPE is not applied to the **Value** vector because:
    - RoPE is designed specifically to encode relative positional information into the attention scores ("How much to attend"), which are determined solely by the dot-product of the **Query** and **Key** vectors.
    - The Value represents the "content" that is being moved, if you rotate the Value you would essentially be changing the content's representation based on where it is in the sequence.


![RoPE Paper Figure 1](../showcase_images/RoPE_paper_figure_1.png)


In [9]:
import easyjupyter
import torch

In [10]:
from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from configs import ConfigTemplate, Scaled_down_text

In [11]:
def precompute_freqs_cis(cfg:ConfigTemplate, head_dim: int, theta: float = 500000.0) -> torch.Tensor:
    """
    Pre-computes the exponential values for RoPE. This is done once during the LLM model initialization def __init__() for the maximum seq_len and cache them.

    Args:
        cfg: Project configurations.
        dim: The dimension of a single attention head (d_model // attn_heads).
        theta: The base frequency for the angle calculations.
    """
    max_seq_len = cfg.text_only.pretrain.annealing_stage.seq_len

    # Calculate the inverse frequencies
    freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=cfg.device)[: (head_dim //2)].float() / head_dim))

    # Create a tensor of sequence positions: [0, 1, 2, ..., max_seq_len-1]
    t = torch.arange(max_seq_len, device=cfg.device, dtype=torch.float32)

    # Outer product to get the angles for each position and dimension, Shape: (max_seq_len, head_dim //2)
    freqs = torch.outer(t, freq)

    # Convert to complex numbers in polar form: magnitude 1, angle freqs, which creates the cosine and sine components inherently 
    return torch.polar(torch.ones_like(freqs), freqs)

In [12]:
def reshape_for_broadcasting(freqs_cis: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """
    Reshape the pre-computed frequencies to match the batch and **H** heads dimensions of the Q or K.
    """
    ndim = x.ndim
    assert 0 <= 1 < ndim
    # Expected x shape: (batch_size, seq_len, attn_heads, head_dim//2)
    assert freqs_cis.shape == (x.shape[1], x.shape[-1]), "freqs_cis shape mismatch!"

    shape = [d if i == 1 or i == ndim - 1 else 1 for i, d in enumerate(x.shape)]
    return freqs_cis.view(*shape)

In [13]:
def apply_rotary_pos_emb(
    q: torch.Tensor, k: torch.Tensor, freqs_cis: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Applies the pre-computed rotations to queries and keys, this is done in the GQA attention layer after projected the X input into Q and K matrices, here we apply the cached freqs_cis to them.

    Args:
        q (torch.Tensor): The queries.
        k (torch.Tensor): The keys.
        freqs_cis (torch.Tensor): The cached freqs_cis.

    Returns:
        tuple[torch.Tensor, torch.Tensor]: The queries and keys with the rotary embeddings applied.
    """
    seq_len = q.shape[1]
    # Slice the freqs_cis to match the current micro-batch sequence length
    freqs_cis_sliced = freqs_cis[:seq_len]

    # Reshape the last dimension into pairs and cast to complex numbers
    q_reshape = torch.view_as_complex(q.float().reshape(*q.shape[:-1], -1, 2))
    k_reshape = torch.view_as_complex(k.float().reshape(*k.shape[:-1], -1, 2))

    # Broadcast the frequencies to match the Q and K shapes
    freqs_cis_broadcast = reshape_for_broadcasting(freqs_cis_sliced, q_reshape)

    # Multiply complex numbers, this causes the rotations to happen
    q_out = torch.view_as_real(q_reshape * freqs_cis_broadcast).flatten(3)
    k_out = torch.view_as_real(k_reshape * freqs_cis_broadcast).flatten(3)

    return q_out.type_as(q), k_out.type_as(k)

In [14]:
# @i-c
# Notebook test
from configs import Scaled_down_text

cfg = Scaled_down_text.initialize()
doc_end_token_id = 2
batch_size = 2
seq_len = 2
attn_heads = 4
d_model = 8
head_dim = d_model // attn_heads
rope_theta = 10_000

# Dummy Query and Key
q = torch.randn(batch_size, seq_len, attn_heads, head_dim, device=cfg.device)
k = torch.randn(batch_size, seq_len, attn_heads, head_dim, device=cfg.device)

# Pre-compute freqs
freqs_cis = precompute_freqs_cis(
    cfg=cfg, head_dim=head_dim, theta=rope_theta
)

# Apply RoPE
q_rot, k_rot = apply_rotary_pos_emb(q, k, freqs_cis)
 


Project Root: /Users/tonyavis/Main/Build_an_LLM
Using config: scaled_down_text


In [15]:
# @i-c
# Test 1: Shape retention
assert q_rot.shape == q.shape, f"Q shape changed: {q.shape} -> {q_rot.shape}"
assert k_rot.shape == k.shape, f"K shape changed: {k.shape} -> {k_rot.shape}"
"Shapes retention passed!"

'Shapes retention passed!'

In [16]:
# @i-c
# Test 2: Values changed for tokens > index 0
assert not torch.allclose(q[:, 1, :,:], q_rot[:, 1, :, :]), f"RoPE failed to rotate tokens at seq_len > 0."
"Rotation applied to subsequent tokens."

'Rotation applied to subsequent tokens.'

In [17]:
# @i-c
# Test 3: Make sure Position 0 is not rotated
assert torch.allclose(q[:, 0, :, :], q_rot[:, 0, :, :], atol=1e-5), "RoPE incorrectly rotated the token at position 0."
"Sequence position 0 correctly remained un-rotated"

'Sequence position 0 correctly remained un-rotated'

Compare again PyTorch's native implementation of RoPE.

In [19]:
# @i-c
from torchtune.modules import RotaryPositionalEmbeddings

max_seq_len = cfg.text_only.pretrain.annealing_stage.seq_len

pytorch_rope = RotaryPositionalEmbeddings(
    dim=head_dim,
    max_seq_len=max_seq_len,
    
    base=cfg.rope_theta,
).to(cfg.device)
q_rot_pytorch = pytorch_rope(q)

In [20]:
# @i-c
difference = torch.abs(q_rot- q_rot_pytorch).max().item()
print(f"Maximum difference between the two implementations: {difference}")
assert torch.allclose(q_rot, q_rot_pytorch, atol=1e-8, rtol=1e-05), "Implementations outputs do not match!"
print("My custom implementations has same similar results as the PyTorch implementation!")

Maximum difference between the two implementations: 2.384185791015625e-07
My custom implementations has same similar results as the PyTorch implementation!
